In [1]:
%pip install "qiskit==2.2.3" "qiskit-machine-learning==0.9.0" "qiskit-optimization==0.7.0" "qiskit-algorithms==0.4.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.0/664.0 kB 20.8 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 54.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 54.4 MB/s  0:00:00
  Created wheel for docplex: filename=docplex-2.32.264-py3-none-any.whl size=703859 sha256=ec5f674bcf84d68d61b01cd3379caf1ddb984603e90ce9b06558131a8193dc9d
  Stored in directory: /root/.cache/pip/wheels/ba/11/de/5d0051be010658975fb2155006b4b8a818136d9f69ed20b4ba
Successfully built docplex
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [qiskit-algorithms]


In [4]:
import sys
import importlib.metadata as metadata
import numpy as np
import scipy
import sklearn
import qiskit
import qiskit_machine_learning
import qiskit_optimization
import qiskit_algorithms

expected = {
    "qiskit": "2.2.3",
    "qiskit-machine-learning": "0.9.0",
    "qiskit-optimization": "0.7.0",
    "qiskit-algorithms": "0.4.0",
}
print("Python:", sys.version.split()[0])
for package in [*expected, "numpy", "scipy", "scikit-learn"]:
    installed = metadata.version(package)
    print(f"{package}: {installed}")
    if package in expected:
        assert installed == expected[package], (
            f"{package} 버전을 확인하고 설치 셀 실행 후 세션을 다시 시작하세요."
        )
print("패키지 import와 버전 확인 완료")

Python: 3.13.18
qiskit: 2.2.3
qiskit-machine-learning: 0.9.0
qiskit-optimization: 0.7.0
qiskit-algorithms: 0.4.0
numpy: 2.4.6
scipy: 1.16.3
scikit-learn: 1.6.1
패키지 import와 버전 확인 완료


In [5]:
import numpy as np
risk = np.array([9,6,8,5], dtype=float)
cost = np.array([5,3,4,2], dtype=float)
for x,s in [(np.array([1,0,1,0]),0),(np.array([1,0,0,0]),1)]:
    f = float((-risk + 0.5*cost) @ x)
    q = f + 20*(x.sum()+s-2)**2
    z = 1-2*x
    print('x:', x, '슬랙:', s, '원래 비용:', f, 'QUBO:', q, 'z:', z)

x: [1 0 1 0] 슬랙: 0 원래 비용: -12.5 QUBO: -12.5 z: [-1  1 -1  1]
x: [1 0 0 0] 슬랙: 1 원래 비용: -6.5 QUBO: -6.5 z: [-1  1  1  1]


In [8]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

gamma = np.pi / 2  # 감마: 비용에 따른 위상 변화 각도, rad
beta = -np.pi / 4  # 베타: 믹서 회전 각도, rad
qc = QuantumCircuit(1)
qc.h(0)
initial = Statevector.from_instruction(qc)
qc.p(-gamma, 0)  # 비용 diag(0,1)에 대한 exp(-i*gamma*C)
after_cost = Statevector.from_instruction(qc)
qc.rx(2 * beta, 0)
after_mixer = Statevector.from_instruction(qc)
print('초기 진폭:', np.round(initial.data, 6))
print('비용 회로 진폭:', np.round(after_cost.data, 6))
print('비용 회로 확률:', np.round(after_cost.probabilities(), 6))
print('믹서 회로 확률:', np.round(after_mixer.probabilities(), 6))
energies = np.array([0.0, 1.0])
print('기대 비용:', float(after_mixer.probabilities() @ energies))
assert np.allclose(after_mixer.probabilities(), [1, 0])

초기 진폭: [0.707107+0.j 0.707107+0.j]
비용 회로 진폭: [0.707107+0.j       0.      -0.707107j]
비용 회로 확률: [0.5 0.5]
믹서 회로 확률: [1. 0.]
기대 비용: 1.3263301508241975e-32


In [7]:
import numpy as np
from itertools import product
from qiskit.quantum_info import Statevector
from qiskit.primitives import StatevectorSampler
from qiskit_algorithms import QAOA
from qiskit_algorithms.optimizers import COBYLA
from qiskit_optimization import QuadraticProgram
from qiskit_optimization.converters import QuadraticProgramToQubo

names = ['서버 A', '서버 B', '서버 C', '서버 D']
risk = np.array([9, 6, 8, 5], dtype=float)
cost = np.array([5, 3, 4, 2], dtype=float)
risk_w, cost_w, max_agents = 1.0, 0.5, 2
# 설치하면 해당 서버의 위험도 점수만큼 감소한다고 가정합니다.
linear = -risk_w * risk + cost_w * cost
qp = QuadraticProgram()
for i in range(4):
    qp.binary_var(f'x{i}')
qp.minimize(linear=linear)
qp.linear_constraint({f'x{i}': 1 for i in range(4)}, '<=', max_agents, 'capacity')
converter = QuadraticProgramToQubo(penalty=20.0)
qubo = converter.convert(qp)
H, offset = qubo.to_ising()
n = H.num_qubits
print('QUBO 변수 순서:', [v.name for v in qubo.variables])
print('QUBO:', qubo.objective)
print('Ising:', H, '상수항:', offset)

# 모든 기저상태에서 QUBO와 Ising 에너지가 같은지 확인합니다.
diagonal = np.real(np.diag(H.to_matrix()))
for k in range(2**n):
    bits = np.array([(k >> j) & 1 for j in range(n)])
    assert np.isclose(qubo.objective.evaluate(bits), diagonal[k] + offset)

qaoa = QAOA(sampler=StatevectorSampler(default_shots=4096, seed=7),
            optimizer=COBYLA(maxiter=160), reps=2,
            initial_point=np.array([0.2, 0.4, 0.6, 0.8]))
energy_scale = max(1.0, float(np.max(np.abs(diagonal))))
result = qaoa.compute_minimum_eigenvalue(H / energy_scale)
# eigenstate의 정수 키에서 낮은 자리부터 변수 순서로 복원합니다.
dist = {int(k, 2) if isinstance(k, str) else int(k): float(p)
        for k, p in result.eigenstate.items()}
# 최종 측정에서 관측된 비트열 중 QUBO 비용이 가장 작은 값을 선택합니다.
best = min((k for k in dist if dist[k] > 0),
           key=lambda k: qubo.objective.evaluate(
               np.array([(k >> j) & 1 for j in range(n)])))
bits = np.array([(best >> j) & 1 for j in range(n)])
x = np.rint(converter.interpret(bits)).astype(int)
feasible = lambda v: int(np.sum(v)) <= max_agents
objective = lambda v: float(linear @ v)
candidates = [np.array(v) for v in product([0, 1], repeat=4) if feasible(v)]
exact = min(objective(v) for v in candidates)
feasible_rate = optimal_rate = 0.0
for k, probability in dist.items():
    b = np.array([(k >> j) & 1 for j in range(n)])
    v = np.rint(converter.interpret(b)).astype(int)
    if feasible(v):
        feasible_rate += probability
        if np.isclose(objective(v), exact):
            optimal_rate += probability

print('선택 비트 [A,B,C,D]:', x.tolist())
print('선택 서버:', [names[i] for i in range(4) if x[i]])
print('위험 감소:', float(risk @ x), '설치 비용:', float(cost @ x))
print('원래 목적함수:', objective(x))
print('설치 개수 제약 충족:', feasible(x))
print('정확 최적값:', exact)
print('정확 최적값과 차이:', objective(x) - exact if feasible(x) else '제약 위반')
print('기대 Ising 에너지:', (float(result.eigenvalue.real) * energy_scale))
print('기대 QUBO 목적값:', (float(result.eigenvalue.real) * energy_scale) + offset)
print('선택 전체 비트 관측 비율:', dist[best])
print('실행 가능한 해 관측 비율:', feasible_rate)
print('최적해 관측 비율:', optimal_rate)
assert np.isclose(sum(dist.values()), 1.0)

고전적 SVM 학습 중...
고전적 SVM 완료

Quantum SVM 학습 시작... 

=== 최종 결과 ===
고전적 SVM 정확도: 0.875
Quantum SVM 정확도: 0.75
학습/시험 크기: (16, 5) (8, 5)
QSVM 분류 결과:
              precision    recall  f1-score   support

           0      0.750     0.750     0.750         4
           1      0.750     0.750     0.750         4

    accuracy                          0.750         8
   macro avg      0.750     0.750     0.750         8
weighted avg      0.750     0.750     0.750         8



서버 순서는 A, B, C, D이며, 위험도는 [9, 6, 8, 5], 설치 비용은 [5, 3, 4, 2]입니다. 두 배열은 교육용 점수이며, cost_w=0.5는 비용 점수를 위험도 점수와 결합하는 환산 계수입니다. x_i=1은 해당 서버에 에이전트를 설치함을 뜻합니다. 설치 시 위험도가 해당 점수만큼 감소한다고 가정합니다. 원래 목적함수의 계수는 [-6.5, -4.5, -6.0, -4.0]이며, 선택된 서버의 계수를 더한 값을 최소화합니다.

입력 데이터 → 이진 변수 → 목적함수·최대 2개 제약 → QUBO → Ising 해밀토니안 → 회로 구성 → 각도 최적화 → 측정 → 서버 선택 복원 순서로 처리합니다. QuadraticProgramToQubo는 남은 설치 가능 개수를 나타내는 슬랙 변수를 추가합니다. 기본 설정에서는 서버 변수 4개와 슬랙 비트 2개를 사용합니다. 출력되는 QUBO 변수 순서를 확인하고, converter.interpret로 업무 변수 4개를 복원합니다.

초기 상태는 6큐비트의 64개 기저상태에 같은 크기의 진폭을 둡니다. 비용 회로는 γ(감마)와 상태별 비용에 따라 위상을 바꿉니다. 이 단계에서는 각 상태의 측정 확률이 유지됩니다. 믹서 회로는 β(베타)를 이용한 X축 회전으로 진폭을 섞습니다. 위상이 다른 진폭의 간섭으로 각 비트열의 확률이 변합니다. reps=2는 비용 회로와 믹서 회로를 두 번 적용하며, 4개 각도를 최적화합니다. Qiskit QAOA의 initial_point는 β(베타) 각도 두 개, γ(감마) 각도 두 개 순서입니다.

COBYLA는 4,096회 측정에서 계산한 기대 에너지를 이용하여 각도를 조정합니다. energy_scale로 해밀토니안을 나누어 계산하고, 출력 단계에서 원래 단위로 환산합니다. 기대 Ising 에너지에 offset을 더하면 기대 QUBO 목적값이 됩니다. 기대값은 모든 관측 결과를 확률로 가중한 평균입니다. 최종 선택의 원래 목적값은 서버 비트로 직접 계산합니다.

정확 최적해는 A와 C를 선택하는 [1,0,1,0]이며 원래 목적값은 -12.5, 위험 감소는 17, 설치 비용 합은 9입니다. 코드는 QAOA의 최종 측정에서 관측된 전체 비트열 중 QUBO 비용이 가장 작은 값을 선택하여 평가합니다. 관측되지 않은 비트열은 선택 후보에 포함하지 않습니다. 실행 가능한 해 비율은 설치 개수가 2 이하인 관측의 비율이며, 최적해 비율은 그중 원래 목적값이 -12.5인 관측의 비율입니다. 제약 위반 결과도 출력에 그대로 드러납니다. 완전탐색은 정확 최적값을 제공하는 검증 기준입니다.